# 🎨 Personal Colour Analysis — Google Colab Runner
**Pipeline**: DeepLabV3 / ClipUNet → K-Means → Munsell Season Classification

**Runtime**: Set to GPU (T4) → Runtime → Change runtime type → T4 GPU

---
### LaPa Dataset Structure
```
data/raw/
  train/
    images/     *.jpg   ← training images
    labels/     *.png   ← segmentation masks (0-10)
    landmarks/  *.txt   ← 106-point landmarks
  val/
    images/     *.jpg   ← validation images  
    landmarks/  *.txt   ← landmarks only  ⚠️ NO LABELS
  test/
    images/     *.jpg   ← test images
    labels/     *.png   ← segmentation masks (used for mIoU eval)
    landmarks/  *.txt   ← landmarks
```
**Important**: LaPa `val` has NO segmentation labels.  
K-Fold CV runs entirely on the `train` split.

In [ ]:
# ── Cell 1: Mount Drive & navigate to project ─────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

# Adjust this path to where you uploaded / cloned the project
PROJECT_DIR = '/content/drive/MyDrive/personal_color'

import os
os.chdir(PROJECT_DIR)
print('Working directory:', os.getcwd())
print('Files:', os.listdir('.'))

In [ ]:
# ── Cell 2: Install dependencies ──────────────────────────────────────────
!pip install -q albumentations scikit-image scikit-learn flask flask-cors pyngrok
!pip install -q git+https://github.com/openai/CLIP.git
print('Dependencies installed ✓')

In [ ]:
# ── Cell 3: Verify dataset structure ──────────────────────────────────────
import sys; sys.path.insert(0, '.')
from src.dataset import dataset_summary
dataset_summary()

# Expected output:
# train   images= ~18K   labels= ~18K   landmarks= ~18K
# val     images= ~2.1K  labels= none   landmarks= ~2.1K  ← NO LABELS
# test    images= ~2.0K  labels= ~2.0K  landmarks= ~2.0K

In [ ]:
# ── Cell 4: Sanity-check model shapes ────────────────────────────────────
import torch
from src.models.system_1_deeplabv3 import DeepLabV3
from src.models.system_2_clipunet  import ClipUNet

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

x = torch.randn(1, 3, 512, 512).to(device)

m1 = DeepLabV3(pretrained=False).to(device)
print('DeepLabV3 output :', m1(x).shape)   # → (1, 11, 512, 512)
del m1; torch.cuda.empty_cache()

m2 = ClipUNet(freeze_clip=True).to(device)
with torch.no_grad():
    print('ClipUNet  output :', m2(x).shape)  # → (1, 11, 512, 512)
del m2; torch.cuda.empty_cache()

In [ ]:
# ── Cell 5a: Train DeepLabV3 — standard (10 % labeled hold-out as monitor) ─
# Val split has no labels → we carve 10 % of train as labeled monitor set.
!python train.py --model deeplab --epochs 50 --device auto

In [ ]:
# ── Cell 5b: Train ClipUNet — K-Fold CV (recommended) ────────────────────
# K-Fold runs on the labeled train split only (val excluded — no labels).
# Produces per-fold checkpoints + kfold_summary.json with avg mIoU.
!python train.py --model clipunet --epochs 30 --kfold --device auto

In [ ]:
# ── Cell 5c: Resume interrupted training ──────────────────────────────────
!python train.py --model clipunet --epochs 30 \
    --resume checkpoints/system_2_clipunet.pth

In [ ]:
# ── Cell 6: Evaluate on TEST split (the only split with labels for eval) ──
!python evaluate.py --model clipunet --mode seg

In [ ]:
# ── Cell 7: Visual comparisons ────────────────────────────────────────────
# Test split: shows original | predicted mask | ground truth mask
!python evaluate.py --model clipunet --mode vis --split test --n 20

# Val split: shows original | predicted mask (no GT available)
# !python evaluate.py --model clipunet --mode vis --split val --n 20

In [ ]:
# ── Cell 8: Display sample visual outputs ─────────────────────────────────
import os
import matplotlib.pyplot as plt
import matplotlib.image as mpimg

vis_dir = 'results/clipunet_vis_test'
samples = sorted(os.listdir(vis_dir))[:4]

fig, axes = plt.subplots(1, len(samples), figsize=(6*len(samples), 5),
                          facecolor='#0d0d0f')
for ax, fname in zip(axes, samples):
    img = mpimg.imread(os.path.join(vis_dir, fname))
    ax.imshow(img); ax.axis('off')
    ax.set_title(fname[:20], color='white', fontsize=8)
plt.tight_layout(); plt.show()

In [ ]:
# ── Cell 9: Full personal-colour pipeline on a portrait ──────────────────
# Upload a portrait to Colab via the Files panel, then:
from google.colab import files
uploaded = files.upload()   # pick a portrait image
portrait_path = list(uploaded.keys())[0]
print(f'Portrait: {portrait_path}')

!python evaluate.py --model clipunet --mode full --img "{portrait_path}"

In [ ]:
# ── Cell 10: Plot training curves ────────────────────────────────────────
import pandas as pd
import matplotlib.pyplot as plt

for model_name in ['deeplab', 'clipunet']:
    log_f = f'results/{model_name}_train_log.csv'
    if not os.path.exists(log_f):
        continue
    df = pd.read_csv(log_f)
    fig, axes = plt.subplots(1, 2, figsize=(12, 4), facecolor='#0d0d0f')
    for ax in axes:
        ax.set_facecolor('#151518')
        ax.tick_params(colors='grey')

    # Loss columns may be train_loss / monitor_loss (standard) or val_loss (kfold)
    loss_col  = 'monitor_loss' if 'monitor_loss' in df.columns else 'val_loss'
    miou_col  = 'monitor_mIoU' if 'monitor_mIoU' in df.columns else 'val_mIoU'

    axes[0].plot(df.epoch, df.train_loss,  color='#c9a96e', label='train')
    axes[0].plot(df.epoch, df[loss_col],   color='#8fbcbb', label='monitor/val',
                 linestyle='--')
    axes[0].set_title(f'{model_name} — Loss', color='white')
    axes[0].legend(facecolor='#151518', labelcolor='white')

    axes[1].plot(df.epoch, df[miou_col], color='#a3be8c', linewidth=1.8)
    axes[1].set_title(f'{model_name} — mIoU', color='white')

    plt.tight_layout(); plt.show()
    print(f'  {model_name} best mIoU = {df[miou_col].max():.4f}')

In [ ]:
# ── Cell 11: Print K-Fold summary ────────────────────────────────────────
import json
for model_name in ['deeplab', 'clipunet']:
    summary_f = f'results/{model_name}_kfold_summary.json'
    if not os.path.exists(summary_f):
        continue
    s = json.load(open(summary_f))
    print(f"\n{model_name.upper()} K-Fold ({s['k_folds']} folds)")
    for i, v in enumerate(s['fold_mious']):
        print(f"  Fold {i+1}: mIoU = {v:.4f}")
    print(f"  Average : {s['average_mIoU']:.4f} ± {s['std_mIoU']:.4f}")

In [ ]:
# ── Cell 12: Launch web demo via ngrok ───────────────────────────────────
import subprocess, threading, time
from pyngrok import ngrok

# Set ngrok auth token (get free token at https://dashboard.ngrok.com/)
# ngrok.set_auth_token('YOUR_TOKEN_HERE')

def run_flask():
    subprocess.run(['python', 'app/app.py'])

t = threading.Thread(target=run_flask, daemon=True)
t.start()
time.sleep(4)

public_url = ngrok.connect(5000)
print(f'\n🌐 Web demo: {public_url}')
print('Open the link above to access the Personal Colour Analysis UI.')